# Homework 6: Decision Trees and Ensemble Learning

This notebook solves the 2026 Homework 6 using the pinned car fuel-efficiency release. It follows the required train/validation/test split, fits the requested tree-based models, and computes each answer from validation results.

Run the cells from top to bottom. The source CSV is read from the course repository and is not modified.

In [2]:
from pathlib import Path
import hashlib
import platform

import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

DATASET_NAME = "car_fuel_efficiency_2026.csv"
EXPECTED_DATASET_SHA256 = "00a5cab178a8b7cd6e9157ee71dabc236ba7f20b1832336d358b07af14ba431f"
TARGET = "fuel_efficiency_mpg"


def find_dataset():
    current = Path.cwd().resolve()
    for directory in (current, *current.parents):
        candidates = (
            directory / "cohorts" / "2026" / "data" / DATASET_NAME,
            directory / "data" / DATASET_NAME,
        )
        for candidate in candidates:
            if candidate.is_file():
                return candidate
    raise FileNotFoundError(
        "Could not find cohorts/2026/data/car_fuel_efficiency_2026.csv"
    )


DATASET_PATH = find_dataset()
dataset_bytes = DATASET_PATH.read_bytes().replace(b"\r\n", b"\n")
dataset_sha256 = hashlib.sha256(dataset_bytes).hexdigest()
assert dataset_sha256 == EXPECTED_DATASET_SHA256, (
    f"Unexpected dataset checksum: {dataset_sha256}"
)

df = pd.read_csv(DATASET_PATH, encoding="utf-8")
assert TARGET in df.columns
print(f"Dataset: {DATASET_PATH}")
print(f"Rows, columns: {df.shape}")
print(f"SHA-256 (normalized line endings): {dataset_sha256}")
print(f"Python: {platform.python_version()} | Pandas: {pd.__version__} | Scikit-Learn: {sklearn.__version__}")
df.head()

Dataset: C:\Users\yanji\Projects\machine-learning-zoomcamp\cohorts\2026\data\car_fuel_efficiency_2026.csv
Rows, columns: (10000, 11)
SHA-256 (normalized line endings): 00a5cab178a8b7cd6e9157ee71dabc236ba7f20b1832336d358b07af14ba431f
Python: 3.11.9 | Pandas: 3.0.6 | Scikit-Learn: 1.9.1


,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


## Prepare the Dataset

Fill missing values with zero, create the specified 60/20/20 train/validation/test split, and vectorize the features with a sparse `DictVectorizer`.

In [3]:
df = df.fillna(0)
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

y_train = df_train.pop(TARGET)
y_val = df_val.pop(TARGET)
y_test = df_test.pop(TARGET)

vectorizer = DictVectorizer(sparse=True)
X_train = vectorizer.fit_transform(df_train.to_dict(orient="records"))
X_val = vectorizer.transform(df_val.to_dict(orient="records"))
X_test = vectorizer.transform(df_test.to_dict(orient="records"))
feature_names = list(vectorizer.get_feature_names_out())


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


print(f"Train: {X_train.shape} | Validation: {X_val.shape} | Test: {X_test.shape}")
print(f"Features after DictVectorizer: {len(feature_names)}")

Train: (6000, 16) | Validation: (2000, 16) | Test: (2000, 16)
Features after DictVectorizer: 16


## Question 1: Decision Tree Regressor

Fit a depth-1 decision tree and map its selected feature back to the original column name.

In [4]:
tree_q1 = DecisionTreeRegressor(max_depth=1, random_state=1)
tree_q1.fit(X_train, y_train)

split_index = tree_q1.tree_.feature[0]
split_feature = feature_names[split_index]
original_feature = split_feature.split("=", 1)[0]
print(f"Encoded split feature: {split_feature}")
print(f"Question 1 answer: {original_feature}")

Encoded split feature: model_year
Question 1 answer: model_year


## Question 2: Random Forest RMSE

Train the requested 10-tree random forest and compute validation RMSE.

In [5]:
forest_q2 = RandomForestRegressor(
    n_estimators=10,
    random_state=1,
    n_jobs=-1,
)
forest_q2.fit(X_train, y_train)
q2_rmse = rmse(y_val, forest_q2.predict(X_val))
print(f"Question 2 validation RMSE: {q2_rmse:.4f}")

Question 2 validation RMSE: 1.8332


## Question 3: Tune the Number of Trees

Evaluate the requested estimator counts on the validation data. The answer selection follows the homework instruction to compare RMSE rounded to three decimal places.

In [6]:
estimator_counts = [10, 50, 100, 150]
q3_scores = {}
for estimator_count in estimator_counts:
    forest = RandomForestRegressor(
        n_estimators=estimator_count,
        random_state=1,
        n_jobs=-1,
    )
    forest.fit(X_train, y_train)
    q3_scores[estimator_count] = rmse(y_val, forest.predict(X_val))

q3_results = pd.DataFrame(
    {
        "n_estimators": list(q3_scores),
        "validation_rmse": list(q3_scores.values()),
    }
)
q3_results["rmse_3dp"] = q3_results["validation_rmse"].round(3)
best_n_estimators = min(
    q3_scores,
    key=lambda value: (round(q3_scores[value], 3), value),
)
display(q3_results)
print(f"Question 3 best n_estimators: {best_n_estimators}")

,n_estimators,validation_rmse,rmse_3dp
0,10,1.833220,1.833
1,50,1.773988,1.774
2,100,1.769786,1.770
3,150,1.769198,1.769


Question 3 best n_estimators: 150


## Question 4: Select `max_depth`

For each requested depth, evaluate all four forest sizes and select the depth with the lowest mean validation RMSE.

In [7]:
depth_values = [10, 15, 20, 25]
q4_rows = []
for depth in depth_values:
    for estimator_count in estimator_counts:
        forest = RandomForestRegressor(
            n_estimators=estimator_count,
            max_depth=depth,
            random_state=1,
            n_jobs=-1,
        )
        forest.fit(X_train, y_train)
        q4_rows.append(
            {
                "max_depth": depth,
                "n_estimators": estimator_count,
                "validation_rmse": rmse(y_val, forest.predict(X_val)),
            }
        )

q4_results = pd.DataFrame(q4_rows)
q4_summary = (
    q4_results.groupby("max_depth", as_index=False)["validation_rmse"]
    .mean()
    .rename(columns={"validation_rmse": "mean_validation_rmse"})
)
best_max_depth = int(
    q4_summary.loc[q4_summary["mean_validation_rmse"].idxmin(), "max_depth"]
)
display(q4_summary)
print(f"Question 4 best max_depth: {best_max_depth}")

,max_depth,mean_validation_rmse
0,10,1.754392
1,15,1.784224
2,20,1.786771
3,25,1.786305


Question 4 best max_depth: 10


## Question 5: Feature Importance

Fit the specified random forest and compare `feature_importances_` for the four candidate original features.

In [8]:
importance_model = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1,
)
importance_model.fit(X_train, y_train)

candidate_features = [
    "vehicle_weight",
    "horsepower",
    "acceleration",
    "engine_displacement",
]
importance_by_feature = dict(
    zip(feature_names, importance_model.feature_importances_)
)
q5_results = pd.DataFrame(
    {
        "feature": candidate_features,
        "importance": [importance_by_feature[name] for name in candidate_features],
    }
).sort_values("importance", ascending=False, ignore_index=True)
most_important_feature = q5_results.loc[0, "feature"]
display(q5_results)
print(f"Question 5 most important feature: {most_important_feature}")

,feature,importance
0,vehicle_weight,0.203313
1,horsepower,0.078271
2,engine_displacement,0.061281
3,acceleration,0.054998


Question 5 most important feature: vehicle_weight


## Question 6: Compare XGBoost Learning Rates

Create training and validation `DMatrix` objects, use the requested watchlist and parameters, and compare validation RMSE for `eta=0.3` and `eta=0.1` over 100 boosting rounds.

In [9]:
import xgboost as xgb

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=feature_names)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=feature_names)
watchlist = [(dtrain, "train"), (dval, "val")]

xgb_params = {
    "max_depth": 6,
    "min_child_weight": 1,
    "objective": "reg:squarederror",
    "nthread": 8,
    "seed": 1,
    "verbosity": 1,
}

eta_scores = {}
for eta in [0.3, 0.1]:
    params = {**xgb_params, "eta": eta}
    model = xgb.train(
        params,
        dtrain,
        num_boost_round=100,
        evals=watchlist,
        verbose_eval=25,
    )
    eta_scores[eta] = rmse(y_val, model.predict(dval))

q6_results = pd.DataFrame(
    {
        "eta": list(eta_scores),
        "validation_rmse": list(eta_scores.values()),
    }
)
if np.isclose(eta_scores[0.3], eta_scores[0.1], rtol=0.0, atol=1e-12):
    best_eta_answer = "Both give equal value"
else:
    best_eta_answer = min(eta_scores, key=eta_scores.get)
display(q6_results)
print(f"Question 6 best eta: {best_eta_answer}")

[0]	train-rmse:2.40653	val-rmse:2.46782
[25]	train-rmse:1.35837	val-rmse:1.74359
[50]	train-rmse:1.18645	val-rmse:1.77517
[75]	train-rmse:1.02727	val-rmse:1.80682
[99]	train-rmse:0.90106	val-rmse:1.82646
[0]	train-rmse:2.74739	val-rmse:2.78635
[25]	train-rmse:1.50918	val-rmse:1.72751
[50]	train-rmse:1.40368	val-rmse:1.71358
[75]	train-rmse:1.35099	val-rmse:1.72057
[99]	train-rmse:1.31192	val-rmse:1.72481


,eta,validation_rmse
0,0.3,1.826458
1,0.1,1.724810


Question 6 best eta: 0.1


## Answer Summary

Collect the answers computed from the pinned 2026 dataset. For Question 2, the notebook shows both the measured RMSE and the nearest listed option.

In [10]:
q2_options = [0.1837, 1.837, 18.37, 183.7]
q2_nearest_option = min(q2_options, key=lambda value: abs(value - q2_rmse))

answer_summary = {
    "Question 1 - split feature": original_feature,
    "Question 2 - validation RMSE": round(q2_rmse, 4),
    "Question 2 - nearest listed option": q2_nearest_option,
    "Question 3 - best n_estimators": best_n_estimators,
    "Question 4 - best max_depth": best_max_depth,
    "Question 5 - most important feature": most_important_feature,
    "Question 6 - best eta": best_eta_answer,
}
answer_summary

{'Question 1 - split feature': 'model_year',
 'Question 2 - validation RMSE': 1.8332,
 'Question 2 - nearest listed option': 1.837,
 'Question 3 - best n_estimators': 150,
 'Question 4 - best max_depth': 10,
 'Question 5 - most important feature': 'vehicle_weight',
 'Question 6 - best eta': 0.1}

## Learning in Public

Review and personalize the draft below before posting. Include `#mlzoomcamp`, tag Alexey Grigorev or DataTalksClub, and remove credentials, private data, and secrets from any screenshot or notebook excerpt.

In [11]:
post_draft = f'''Completed Module 6 of Machine Learning Zoomcamp 2026!

I compared decision trees, random forests, and XGBoost for car fuel-efficiency regression, then tuned tree depth, forest size, and learning rate.

My repo: <LINK>

#mlzoomcamp @Alexey Grigorev'''
print(post_draft)
print("\nSubmit the homework results at: https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw06")

Completed Module 6 of Machine Learning Zoomcamp 2026!

I compared decision trees, random forests, and XGBoost for car fuel-efficiency regression, then tuned tree depth, forest size, and learning rate.

My repo: <LINK>

#mlzoomcamp @Alexey Grigorev

Submit the homework results at: https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw06
